In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(r"C:\Users\Junayed\pandas_prac\Aug_26\messy_vetclinic.csv")

In [3]:
df.head(5)

,PatientID,PetName,Species,Breed,OwnerName,OwnerPhone,VisitDate,AgeYears,WeightKg,Vaccinated,VisitReason,Cost,Notes
0,V001,Buddy,Dog,Labrador,Sarah Kim,(415) 555-0142,2024-03-11,4.0,28.5,Yes,Checkup,65.00,NaN
1,V002,MAX,dog,Poodle,james ortiz,415-555-0198,03/12/2024,7.0,12.3,yes,Vaccination,45.00,NaN
2,V003,Bella,Cat,Siamese,Maria Chen,(628) 555-0110,2024-03-12,2.0,4.1,No,Grooming,40.00,NaN
3,V004,Charlie,CAT,Domestic Shorthair,Tom Reyes,6285550177,12-Mar-2024,1.0,3.8 kg,N,Vaccination,45.00,NaN
4,V005,Daisy,Rabbit,Holland Lop,Priya Nair,(510) 555-0199,2024-03-13,-1.0,1.9,Yes,Checkup,55.00,Owner unsure of age


In [4]:
df["PetName"] = df["PetName"].str.strip().str.title()

In [5]:
df["Species"].value_counts()

Species
Dog       17
Cat        9
Rabbit     5
cat        4
Bird       3
dog        1
CAT        1
RABBIT     1
cat        1
Name: count, dtype: int64

In [6]:
df["Species"] = df["Species"].str.strip().str.title()

In [7]:
df["Species"].value_counts()

Species
Dog       18
Cat       15
Rabbit     6
Bird       3
Name: count, dtype: int64

In [8]:
df["Breed"] = df["Breed"].str.strip().str.title()

In [9]:
df["OwnerName"] = df["OwnerName"].str.strip().str.title()

In [10]:
df["OwnerPhone"] = df["OwnerPhone"].str.replace(r"[()-. ]", "", regex=True)
def clean_number(val):
    if pd.isna(val):
        return np.nan
    elif len(val) == 10:
        return f"{val[0:3]}-{val[3:6]}-{val[6:10]}"
    else:
        return val

df["OwnerPhone"] = df["OwnerPhone"].apply(clean_number)
df[["PatientID", "OwnerPhone"]]

,PatientID,OwnerPhone
0,V001,415-555-0142
1,V002,415-555-0198
2,V003,628-555-0110
3,V004,628-555-0177
4,V005,510-555-0199
5,V006,510-555-0134
6,V007,415-555-0177
7,V008,415-555-01O0
8,V009,628-555-0155
9,V010,NaN


In [11]:
df.loc[df["OwnerPhone"].isna(), "Notes"] = "Update Phone no."

In [12]:
df["VisitDate"] = pd.to_datetime(df["VisitDate"], format='mixed', errors='coerce')
df["VisitDate"]

0    2024-03-11
1    2024-03-12
2    2024-03-12
3    2024-03-12
4    2024-03-13
5    2024-03-13
6    2024-03-14
7    2024-03-14
8    2024-03-15
9    2024-03-15
10   2024-03-15
11   2024-03-15
12   2024-03-16
13   2024-03-16
14   2024-03-16
15   2024-03-17
16   2024-03-17
17   2024-03-18
18   2024-03-18
19   2024-03-19
20   2024-03-19
21   2024-03-19
22   2024-03-20
23   2024-03-20
24   2024-03-21
25   2024-03-21
26   2024-03-21
27   2024-03-22
28   2024-03-22
29   2024-03-23
30   2024-03-23
31   2024-03-24
32   2024-03-24
33   2024-03-25
34   2024-03-25
35   2024-03-26
36   2024-03-26
37   2024-03-27
38   2024-03-27
39   2024-03-28
40   2024-03-28
41   2024-03-29
Name: VisitDate, dtype: datetime64[us]

In [13]:
df.loc[df["AgeYears"] < 0, ["PatientID", "PetName", "Species", "AgeYears", "Notes"]]

,PatientID,PetName,Species,AgeYears,Notes
4,V005,Daisy,Rabbit,-1.0,Owner unsure of age
19,V020,Gizmo,Bird,-1.0,NaN


In [14]:
df.loc[df["AgeYears"] > 30, ["PatientID", "PetName", "Species", "AgeYears", "Notes"]]

,PatientID,PetName,Species,AgeYears,Notes
15,V016,Ruby,Dog,99.0,"Age looks off, verify with owner"


In [15]:
df.loc[(df["AgeYears"] < 0) | (df["AgeYears"] > 30), "AgeYears"] = np.nan

In [20]:
df["WeightKg"] = df["WeightKg"].str.rstrip("kg").str.strip()
df["WeightKg"] = df["WeightKg"].astype(float)
df["WeightKg"].describe()

count     41.000000
mean      58.899268
std      215.878966
min        0.030000
25%        3.800000
50%        4.900000
75%       18.300000
max      999.000000
Name: WeightKg, dtype: float64

In [21]:
df.loc[df["WeightKg"] > 60, ["PatientID", "PetName", "Species", "WeightKg", "Notes"]]

,PatientID,PetName,Species,WeightKg,Notes
5,V006,Rocky,Dog,999.0,Weight likely a data entry error
16,V017,Bear,Dog,999.0,Scale malfunction noted by tech


In [22]:
df["WeightKg"] = df["WeightKg"].clip(upper=60)
df.loc[[5,16], ["PatientID", "PetName", "Species", "WeightKg", "Notes"]]

,PatientID,PetName,Species,WeightKg,Notes
5,V006,Rocky,Dog,60.0,Weight likely a data entry error
16,V017,Bear,Dog,60.0,Scale malfunction noted by tech


In [23]:
df["Vaccinated"].value_counts()

Vaccinated
Yes    21
No     13
yes     3
N       2
1       1
0       1
Y       1
Name: count, dtype: int64

In [24]:
vacine_map = {
    "yes": "Yes",
    "N": "No",
    "1": "Yes",
    "0": "No",
    "Y": "Yes"
}

df["Vaccinated"] = df["Vaccinated"].replace(vacine_map)
df["Vaccinated"].value_counts()

Vaccinated
Yes    26
No     16
Name: count, dtype: int64

In [26]:
df["VisitReason"] = df["VisitReason"].str.title()

In [27]:
df["VisitReason"].value_counts()

VisitReason
Checkup        18
Vaccination    10
Grooming        7
Surgery         6
Chekup          1
Name: count, dtype: int64

In [28]:
reason_fix = {"Chekup": "Checkup"}
df["VisitReason"].replace(reason_fix)
df["VisitReason"] = df["VisitReason"].str.strip() 

In [29]:
df["Cost"] = df["Cost"].astype(str).str.replace(",", "", regex=False)
df["Cost"] = pd.to_numeric(df["Cost"], errors="coerce")
df["Cost"].describe()

count      42.000000
mean      223.476190
std       438.928809
min        15.000000
25%        40.000000
50%        47.500000
75%        65.000000
max      1500.000000
Name: Cost, dtype: float64